## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [5]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

import os
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [6]:
load_dotenv(override=True)
ADESSO_BASE_URL = os.getenv('ADESSO_BASE_URL')
adesso_sovereign_ai_hub_key = os.getenv('ADESSO_SOVEREIGN_AI_HUB_KEY')
adesso_api_key = os.getenv('ADESSO_API_KEY')
vultr_api_key = os.getenv('VULTR_API_KEY')

model_name = os.getenv('FREE_DEFAULT_MODEL')
adesso = OpenAI(base_url=ADESSO_BASE_URL, api_key=adesso_sovereign_ai_hub_key)
# openai = OpenAI()


In [7]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [8]:
print(linkedin)

   
Contact
er.singhathia@gmail.com
www.linkedin.com/in/msinghathia
(LinkedIn)
Top Skills
CMDB
Now Assist (GenAI)
ServiceNow Flow Designer
Languages
Punjabi (Native or Bilingual)
French (Elementary)
English (Full Professional)
Hindi (Native or Bilingual)
Certifications
Microsoft Certified: Azure Data
Fundamentals
Prepare for the Red Hat Certified
System Administrator (EX200) Exam
ServiceNow Certified System
Administrator
Microsoft Certified: Azure
Fundamentals (AZ-900)
Manpreet Singhathia
Certified ServiceNow Developer | CSA | CAD | CIS-HR | CIS-DF |
HRSD & ITSM Specialist | 6+ Years IT Experience
India
Summary
Senior ServiceNow Developer | Specialist in Service Catalog, ITSM,
HRSD &amp; AI (Now Assist)I am a Certified ServiceNow Developer
at adesso with 6+ years of IT experience. I specialize in building
high-performance, user-centric solutions, with a deep expertise in
Service Catalog design, Flow Designer automation, and AI-driven
platform modernizationCore Technical Strengths:Servi

In [10]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [11]:
name = "Manpreet Singhathia"

In [12]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [13]:
system_prompt

'You are acting as Manpreet Singhathia. You are answering questions on Manpreet Singhathia\'s website, particularly questions related to Manpreet Singhathia\'s career, background, skills and experience. Your responsibility is to represent Manpreet Singhathia for interactions on the website as faithfully as possible. You are given a summary of Manpreet Singhathia\'s background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don\'t know the answer, say so.\n\n## Summary:\nMy name is Manpreet Singhathia. I am a Senior ServiceNow Developer with 6+ years of IT experience, specializing in Service Catalog, ITSM, HRSD, automation, integrations, and AI-driven modernization\nOutside of work, I love traveling, hiking, road trips, and exploring new tech as a tech geek\n\n## LinkedIn Profile:\n\xa0 \xa0\nContact\ner.singhathia@gmail.com\nwww.linkedin.com/in/msinghathi

In [14]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = adesso.chat.completions.create(model=model_name, messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [15]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [16]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [51]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [52]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [58]:
vultr_base_url = os.getenv('VULTR_BASE_URL')
vultr = OpenAI(
    api_key=vultr_api_key, 
    base_url=vultr_base_url
)

adesso_premium = OpenAI(base_url=ADESSO_BASE_URL, api_key=adesso_api_key)

model_name = os.getenv('PAID_ESCALATION_MODEL')

In [74]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = vultr.chat.completions.parse(model="nvidia/DeepSeek-V3.2-NVFP4", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [68]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = adesso.chat.completions.create(model="qwen-3.6-35b-sovereign", messages=messages)
reply = response.choices[0].message.content

In [69]:
reply

"\n\nThank you for asking! Based on my current background and professional track record, I don't hold any patents. My career has been focused on architecting, building, and optimizing enterprise-grade ServiceNow solutions, driving workflow automation, and pioneering AI-driven platform modernization for clients across ITSM, HRSD, and Service Catalog.\n\nThat said, I'm a strong advocate for continuous innovation and process optimization. In my recent roles, I've consistently introduced scalable solutions and automation frameworks that have cut manual fulfillment time by over 40% and boosted agent resolution efficiency by 67% through GenAI integration. If you're interested in diving into my technical implementations, recent project highlights, or how I approach platform modernization and integration architecture, I'd be glad to walk you through it!"

In [70]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, feedback='The response directly answers the question (no patents) and maintains a professional, engaging tone while offering to discuss further details, which aligns with the instructions.')

In [71]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = adesso.chat.completions.create(model="qwen-3.6-35b-sovereign", messages=messages)
    return response.choices[0].message.content

In [72]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = adesso.chat.completions.create(model="qwen-3.6-35b-sovereign", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [75]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Passed evaluation - returning reply
Failed evaluation - retrying
The response is unacceptable because it is written entirely in Pig Latin, which makes it unprofessional, difficult to read, and not engaging for a potential client or employer. The content is relevant (acknowledges no patents, focuses on ServiceNow development), but the format completely undermines the professional tone expected from an agent representing Manpreet Singhathia. The response should be rewritten in clear, professional English to properly convey the message while maintaining the engaging and client-focused tone requested.
Passed evaluation - returning reply
